# Final Submission Notebook

This is the shortest notebook we would use to walk someone through the final datathon package. It rebuilds the submission files, checks that they pass validation, and shows the required CSV outputs.

## 1. Build And Validate The Submission Package

In [1]:
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Markdown, display

ROOT = Path('../').resolve()
SUBMISSION_DIR = ROOT / 'data' / 'submission'
MAP_PATH = ROOT / 'maps' / 'proposed_charging_network.html'
EXPLORER_PATH = ROOT / 'maps' / 'offline_scenario_explorer.html'

commands = [
    [sys.executable, str(ROOT / 'scripts' / 'run_pipeline.py')],
    [sys.executable, str(ROOT / 'scripts' / 'generate_submission_package.py')],
    [sys.executable, str(ROOT / 'scripts' / 'validate_submission.py')],
    [sys.executable, str(ROOT / 'scripts' / 'build_offline_scenario_explorer.py')],
]

env = os.environ.copy()
existing_pythonpath = env.get('PYTHONPATH', '')
env['PYTHONPATH'] = str(ROOT) if not existing_pythonpath else f"{ROOT}:{existing_pythonpath}"

for command in commands:
    print(f"\n$ {' '.join(command)}")
    subprocess.run(command, cwd=ROOT, check=True, env=env)

print('\nSubmission package rebuilt and validated successfully.')


$ python scripts/run_pipeline.py


RTIG ROADS DATATHON PIPELINE - FULL RUN

[1/3] Downloading raw data from ArcGIS REST API...
Using cached file: data/raw/carreteras_RTIG.geojson
Cached features available: 1602
✓ Downloaded 1602 road segments to data/raw/carreteras_RTIG.geojson

[2/3] Preprocessing and feature engineering...
Loading data from data/raw/carreteras_RTIG.geojson...
Loaded 1602 valid features

Engineering features...

Standardizing data...

Saving processed data...
Saved GeoDataFrame: data/processed/roads_processed_gdf.parquet
Saved cleaned data: data/processed/roads_processed.parquet
Saved CSV: data/processed/roads_processed.csv
Saved GeoJSON: data/processed/roads_processed.geojson

Converting to Polars...
Polars DataFrame shape: (1602, 22)
Polars version available at data/processed/roads_processed.parquet
✓ Processed 1602 features
✓ Generated 23 columns with engineered features

[3/3] Pipeline Summary:
  - Total road segments: 1602
  - Total network length: 38529 km
  - TEN-T corridors: 535
  - Data qualit

Saved: data/submission/File 1.csv
Saved: data/submission/File 2.csv
Saved: data/submission/File 3.csv
Saved: maps/proposed_charging_network.html

$ python scripts/validate_submission.py


Submission validation passed.

$ python scripts/build_offline_scenario_explorer.py


Saved: maps/offline_scenario_explorer.html

Submission package rebuilt and validated successfully.


## 2. Preview The Required CSV Deliverables

In [2]:
file_1 = pd.read_csv(SUBMISSION_DIR / 'File 1.csv')
file_2 = pd.read_csv(SUBMISSION_DIR / 'File 2.csv')
file_3 = pd.read_csv(SUBMISSION_DIR / 'File 3.csv')

display(Markdown('### File 1.csv'))
display(file_1)

display(Markdown('### File 2.csv'))
display(file_2.head(10))
print(f'File 2 rows: {len(file_2)}')

display(Markdown('### File 3.csv'))
display(file_3.head(10))
print(f'File 3 rows: {len(file_3)}')

### File 1.csv

,total_proposed_stations,total_existing_stations_baseline,total_friction_points,total_ev_projected_2027
0,342,0,113,400000


### File 2.csv

,location_id,latitude,longitude,route_segment,n_chargers_proposed,grid_status
0,IBE_001,41.994271,2.794496,AP-7N,8,Moderate
1,IBE_002,41.425589,1.830648,AP-7N,8,Moderate
2,IBE_003,40.874855,0.787201,AP-7N,8,Moderate
3,IBE_004,40.042986,0.024942,AP-7N,8,Moderate
4,IBE_005,39.229117,-0.413595,AP-7N,8,Moderate
5,IBE_006,38.517535,-0.257025,AP-7N,8,Moderate
6,IBE_007,37.802077,-0.849010,AP-7N,8,Moderate
7,IBE_008,37.315278,-1.784754,AP-7N,8,Moderate
8,IBE_009,36.105964,-5.708024,N-340,8,Moderate
9,IBE_010,36.293733,-5.294603,N-340,8,Moderate


File 2 rows: 342


### File 3.csv

,bottleneck_id,latitude,longitude,route_segment,distributor_network,estimated_demand_kw,grid_status
0,FRIC_001,41.994271,2.794496,AP-7N,Endesa,1200,Moderate
1,FRIC_002,41.425589,1.830648,AP-7N,Endesa,1200,Moderate
2,FRIC_003,40.874855,0.787201,AP-7N,Endesa,1200,Moderate
3,FRIC_004,40.042986,0.024942,AP-7N,i-DE,1200,Moderate
4,FRIC_005,39.229117,-0.413595,AP-7N,i-DE,1200,Moderate
5,FRIC_006,38.517535,-0.257025,AP-7N,i-DE,1200,Moderate
6,FRIC_007,37.802077,-0.849010,AP-7N,Endesa,1200,Moderate
7,FRIC_008,37.315278,-1.784754,AP-7N,Endesa,1200,Moderate
8,FRIC_009,36.105964,-5.708024,N-340,Endesa,1200,Moderate
9,FRIC_010,36.293733,-5.294603,N-340,Endesa,1200,Moderate


File 3 rows: 113


## 3. Show Assumptions And Readiness Notes

In [3]:
assumptions_path = SUBMISSION_DIR / 'ASSUMPTIONS.md'
display(Markdown(assumptions_path.read_text(encoding='utf-8')))

summary_path = ROOT / 'docs' / 'executive_summary.md'
display(Markdown(summary_path.read_text(encoding='utf-8')))

# Submission Assumptions

- `File 1.csv`, `File 2.csv`, and `File 3.csv` were generated from the local RTIG dataset.
- Station placement uses a spacing heuristic of `120.0` km along A-/AP-/N- corridors using PK ranges.
- Grid status is currently a proxy classification to preserve submission structure until real distributor capacity nodes are added.
- Distributor assignment is currently a geographic approximation for structural compliance only.
- Existing-station baseline source status: `missing:existing_interurban_stations.csv`.
- EV projection source status: `missing:ev_projection_2027.csv`.

Before final submission, replace proxy values with:
1. National Access Point charging baseline filtered to interurban roads.
2. The mandatory datos.gob.es EV projection output for 2027.
3. i-DE / Endesa / Viesgo node-level capacity matching and documented thresholds.


# Executive Summary
## IE Iberdrola Datathon March 2026

## 1. Problem Framing

The goal of this project is to help Iberdrola decide where new public fast-charging stations should be prioritized on Spain's interurban road network for a 2027 scenario. The challenge is not only geographic coverage. A location can look good from a mobility point of view and still be a bad choice if the nearby grid cannot support high-power chargers.

Our approach focuses on three questions:

1. Which interurban corridors matter most for a first charging rollout?
2. How can we propose a network with as few stations as possible while still covering long-distance travel?
3. Where do mobility needs and grid limitations collide, creating friction points that need reinforcement or phased deployment?

## 2. Data Used

The current repository is built around the Ministry of Transport RTIG road dataset retrieved through the ArcGIS REST service. After cleaning and reprojection, the working dataset contains 1,602 road segments and roughly 38,529 km of network length.

Main fields used in the current version:

- road identifier (`carretera`)
- PK start and end (`pk_inicio`, `pk_fin`)
- road type (`tipo_de_via`)
- TEN-T membership (`tent`)
- geometry and derived length

The project structure is ready to integrate the other required datathon sources:

- national charging-station baseline
- 2027 EV projection from the mandatory datos.gob.es exercise
- i-DE, Endesa, and Viesgo grid-capacity datasets

At the moment, the road-network component is real and reproducible. Some charging-demand and grid-classification values are still provisional and are explicitly documented in [ASSUMPTIONS.md](data/submission/ASSUMPTIONS.md).

## 3. Methodology

We used the following workflow:

1. Download RTIG road geometries from the Ministry REST API and keep a local cached copy so the pipeline can still run if the service is unavailable.
2. Clean the Esri geometry payload, validate features, and reproject the network to WGS84 for mapping.
3. Filter the network to interurban corridors relevant to the brief, especially A-, AP-, and N- roads.
4. Use PK ranges and route lengths as a first heuristic for proposed station spacing.
5. Generate the required output structure:
   - `File 1.csv`: global KPI summary
   - `File 2.csv`: proposed charging locations
   - `File 3.csv`: friction points
6. Validate the final files against the datathon rules before submission.

This gives a reproducible baseline package even before all external datasets are added.

## 4. Current Results

The current generated submission package contains:

- 342 proposed charging locations in `File 2.csv`
- 113 friction points in `File 3.csv`
- a self-contained HTML map in `maps/proposed_charging_network.html`

These values should be interpreted as a structured baseline rather than final business numbers. Their main value today is that they make the project submission-safe: the required files exist, they pass validation, and the methodology is traceable.

## 5. Key Findings

Several patterns are already visible from the current road-network analysis:

- TEN-T corridors account for a meaningful share of strategic interurban connectivity and are a natural starting point for a phased rollout.
- Long interurban axes such as AP-7 and other national corridors are the most likely candidates for early deployment because they combine long-distance travel relevance with clearer spacing logic.
- A station-placement heuristic based on route span immediately produces a set of friction points, which is useful for discussing grid reinforcement and phasing even before full distributor-node matching is complete.

## 6. Main Assumptions And Limits

This is the most important section for the jury.

The current repository is strong on reproducibility and road-network structure, but there are still three assumptions that must be replaced before final submission if we want to maximize the score:

1. `total_existing_stations_baseline` still uses a placeholder default when the national baseline dataset is not present locally.
2. `total_ev_projected_2027` still uses a placeholder default when the mandatory EV forecast output is not present locally.
3. `grid_status` and distributor assignment are still proxy values until the real i-DE, Endesa, and Viesgo capacity-node data is integrated.

These limitations are already declared in the repo and should also be stated clearly in the final presentation. Hiding them would be a mistake. Explaining them well is much stronger.

## 7. Recommended 2027 Rollout Logic

Based on the current work, the most defensible rollout logic is:

### Phase 1

Focus on the major interurban corridors with the clearest long-distance need and strongest strategic relevance, especially TEN-T-aligned axes.

### Phase 2

Refine the candidate list using the real baseline charging map and the 2027 EV projection so charger counts can be justified more credibly by route.

### Phase 3

Overlay real distributor capacity nodes and separate stations into:

- immediately viable
- viable with moderate upgrades
- delayed pending reinforcement

This gives Iberdrola a practical way to distinguish between "best sites to build now" and "best sites to protect for later expansion."

## 8. Why This Matters For Iberdrola

From a business point of view, the project is useful because it translates a broad electrification challenge into a shortlist of deployable locations and a second shortlist of grid bottlenecks. That is a more actionable output than a simple map of roads or a generic demand forecast.

Even in its current state, the repo already helps answer:

- where a first rollout could start
- how to structure a minimum viable national corridor network
- which locations need a grid conversation before a charging conversation

## 9. What Would Most Improve The Final Score

If we had to prioritize only a few remaining tasks before submission, they would be:

1. integrate the mandatory EV projection result from the datos.gob.es exercise
2. integrate the interurban charging baseline from the National Access Point
3. replace proxy grid-status logic with nearest-substation matching and documented thresholds
4. present the final numbers in one notebook with visible outputs and one short, clear pitch

## 10. Final Assessment

The strongest part of this project today is not that it already solves the entire problem perfectly. It is that it now has a reproducible structure, compliant output files, a clear map, and an honest framework for adding the missing high-value data sources.

That makes it a credible datathon submission. To become a finalist-level one, the next step is to improve the realism of demand and grid viability, not to add more software complexity.


## 4. Open The Final Artifacts

In [4]:
display(HTML(f"<p><a href='{MAP_PATH.as_posix()}' target='_blank'>Open proposed charging network map</a></p>"))
display(HTML(f"<p><a href='{EXPLORER_PATH.as_posix()}' target='_blank'>Open offline scenario explorer</a></p>"))
print(MAP_PATH)
print(EXPLORER_PATH)

maps/proposed_charging_network.html
maps/offline_scenario_explorer.html
